**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/zhai-2025"], check=True)
    os.chdir("/content/encore/entries/zhai-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Zhai et al. (2025) from dandiset 001538

Paper: Zhai, Cui, Wokosin, Sun, Tkatch, Crittenden, Graybiel and Surmeier (2025), "State-dependent
modulation of spiny projection neurons controls levodopa-induced dyskinesia in a mouse model of
Parkinson's disease", Science Advances, doi:10.1126/sciadv.adv8224.

Dataset: [DANDI:001538](https://dandiarchive.org/dandiset/001538/0.261124.1858), version
0.261124.1858, streamed from the archive. This version was asked for by the run that made this
attempt; it is also the newest published version (the others are 0.260527.1302 and the draft). The
paper cites no version of the dandiset.

Claims: `claims/zhai-2025.yaml`, written from the version of record in PubMed Central (PMC12674127),
its figure legends and Methods. The paper's own per-cell and per-mouse tables are read from Zenodo
record 17307059 (doi:10.5281/zenodo.17307059, the latest version of 10.5281/zenodo.14589035; file
"Zhai et al. 2025 Datasets.zip", md5 112da5b1b00208262bd7887dd51b54ee). They are used for the link
claims and for the claims on data that are not in the dandiset (`data: elsewhere`). The paper's
analysis code (Zenodo 14145776, 14163502 and 14163731) was read to fix the measures; see the
docstring of `zhai.py`.

This is a repeat attempt made blind: no earlier attempt on this dataset was seen.

| Claim | Data | Tested here |
|---|---|---|
| link-somatic-cells | dandiset and paper tables | yes |
| link-dendritic-cells | dandiset and paper tables | yes |
| link-sr-cells | dandiset and paper tables | yes |
| link-grab-rois | dandiset and paper tables | yes |
| link-aim-mice | dandiset and paper tables | yes |
| fig1f-dspn-rheobase-on-lower, fig1e-dspn-fi-on-higher, fig1f-dspn-sch-no-change | dandiset | yes |
| fig1i-dspn-dendritic-on-higher, fig1i-dspn-dendritic-sch-lower | dandiset | yes |
| fig1-dspn-lesion-and-off-somatic, fig1-dspn-lesion-dendritic-unchanged | paper tables (elsewhere) | yes |
| fig2h-dspn-sr-amplitude-on-larger, fig2i-dspn-sr-frequency-unchanged | dandiset | yes |
| fig2bc-dspn-spines | dandiset (raw image stacks) | blocked, not runnable |
| fig3d-ispn-rheobase-on-higher, fig3c-ispn-fi-on-lower, fig3d-ispn-sul-no-change | dandiset | yes |
| fig3f-ispn-dendritic-on-lower, fig3f-ispn-dendritic-sul-no-change | dandiset | yes |
| fig3-ispn-lesion-and-off-somatic, fig3-ispn-lesion-dendritic-up | paper tables (elsewhere) | yes |
| fig4g-ispn-sr-amplitude-on-smaller, fig4h-ispn-sr-frequency-unchanged | dandiset | yes |
| fig4bc-ispn-spines, fig4k-confocal-spines | dandiset | blocked |
| fig5f-ach-off-elevated, fig5f-ach-off-like-6ohda, fig5f-ach-6ohda-elevated, fig5f-ach-d2-control, fig5f-ach-da-suppresses | dandiset | yes |
| fig6c-m1r-antag-somatic-unchanged, fig6d-m1r-antag-dendritic-lower | dandiset | yes |
| fig6f-m1r-antag-spines, fig7h-cdgi-ko-spines, fig9f-m1r-crispr-spines | dandiset (raw image stacks) | blocked, not runnable |
| fig7c-cdgi-ko-somatic-oscillation, fig7f-cdgi-ko-dendritic-unchanged | dandiset | yes |
| fig7e-oxom-cdgi | paper tables (elsewhere) | yes |
| fig8a-cdgi-rotations, fig10a-m1r-rotations, fig10ef-open-field | paper tables (elsewhere) | yes |
| fig8bc-cdgi-aims, fig10b-m1r-aims | dandiset with labels | yes |
| fig9d-m1r-crispr-somatic-unchanged | dandiset | yes |

Statistics follow the paper's Methods ("Data acquisition and statistical analysis"): two-sided
Mann-Whitney tests for unpaired data and Wilcoxon tests for paired data, P < 0.05. SciPy's
default method is used for both (exact P for small samples without ties, the normal
approximation otherwise), a standard choice. No analysis here uses random draws, so no seed is
needed.

In [ ]:
import time, resource, warnings
import numpy as np
import pandas as pd
from scipy import stats
import zhai

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
print("dandiset", zhai.DANDISET, "version", zhai.VERSION)

In [ ]:
t0 = time.time()
assets = zhai.list_assets()
ice = [r for _, r in assets[assets.kind.isin(["icephys", "icephys+ophys"])].iterrows()]
res = zhai.pmap(zhai.icephys_file, ice, workers=8)
som_sweeps = pd.concat([r[0] for r in res], ignore_index=True)
den_pairs = pd.concat([r[1] for r in res], ignore_index=True)
print(f"current-clamp files: {len(ice)}, {time.time() - t0:.0f} s")
sr_sweeps = pd.concat(zhai.pmap(zhai.sr_file, [r for r in ice if r.kind == "icephys"], workers=8), ignore_index=True)
print(f"Sr2+-oEPSC sweeps read, {time.time() - t0:.0f} s")
grab_trials = pd.concat(zhai.pmap(zhai.grab_file, [r for _, r in assets[assets.kind == "ophys"].iterrows()], workers=4),
                        ignore_index=True)
print(f"GRAB-ACh3.0 trials read, {time.time() - t0:.0f} s")
aim_assets = assets[assets.nwb & (assets.cell_type == "pan") & assets.kind.isin(["", "image"])]
aim = pd.concat(zhai.pmap(zhai.aim_file, [r for _, r in aim_assets.iterrows()], workers=16), ignore_index=True)
T = zhai.paper_tables()
print(f"all loaded in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

What the files hold, by kind, and the counts by cell type and condition. Groups are formed from
the `condition` column of each file's intracellular recordings table (one value per sweep), as
the claims file asks, not from the file name: files named for a drug also hold baseline sweeps.

In [ ]:
print("assets:", len(assets), "NWB:", int(assets.nwb.sum()), "other (videos):", int((~assets.nwb).sum()))
print(assets[assets.nwb].groupby(["kind", "cell_type", "group", "genotype"]).size().rename("files").to_string())

som_cells = zhai.per_cell_somatic(som_sweeps)
den_cells = zhai.per_cell_dendritic(den_pairs)
print("\nSomatic current-step cells by cell type and condition:")
print(som_cells.groupby(["cell_type", "condition"]).agg(cells=("cell", "size"), files=("path", "nunique")).to_string())
print("\nDendritic cells (pairs with basal ratio 0.5 to 2, averaged):")
print(den_cells.groupby(["cell_type", "condition"]).agg(cells=("index", "size"), pairs=("n_pairs", "sum")).to_string())

srt = sr_sweeps.groupby(["cell_type", "path", "cell", "condition"]).size().rename("sweeps").reset_index()
print("\nSr2+-oEPSC cells:")
print(srt.groupby(["cell_type", "condition"]).agg(cells=("cell", "size"), sweeps_min=("sweeps", "min"), sweeps_max=("sweeps", "max")).to_string())
print("\nGRAB-ACh3.0 regions by group:", grab_trials.groupby("group").apply(lambda d: d[["path", "region"]].drop_duplicates().shape[0]).to_dict())
print("AIM mice by figure and group:", aim.groupby(["figure", "group_label"]).mouse.nunique().to_dict())

print("\nUnits and ranges")
print(f"somatic membrane potential: {som_sweeps.vmin.min():.1f} to {som_sweeps.vmax.max():.1f} mV (volts in the files, scaled by each series' conversion)")
print("per file minimum and maximum (mV):", som_sweeps.groupby("path").vmin.min().round(0).agg(["min", "max"]).tolist(), som_sweeps.groupby("path").vmax.max().round(0).agg(["min", "max"]).tolist())
print("somatic step currents (pA):", sorted(som_sweeps.step_pA.unique().astype(int).tolist()))
print("somatic sweep length (samples at 10 kHz):", sorted(som_sweeps.n_samples.unique().tolist()))
print("line-scan basal ratio range:", den_pairs.basal_ratio.round(3).agg(["min", "max"]).tolist(), "; pairs outside 0.5 to 2:", int(((den_pairs.basal_ratio < 0.5) | (den_pairs.basal_ratio > 2)).sum()), "of", len(den_pairs))
print("Sr2+ sampling rates (Hz):", sorted(sr_sweeps.fs.round(0).unique().tolist()), "; flash time in sweep (s):", sorted(sr_sweeps.flash_s.round(4).unique().tolist()))
print("GRAB frame interval (s):", grab_trials.frame_dt.round(4).agg(["min", "max"]).tolist(), "(the paper gives 5.01 frames per second)")
print("AIM single scores range:", aim[aim.behavior != "total"].score.agg(["min", "max"]).tolist(), "(scale 0 to 4)")

### Clocks

Each check contrasts the response in the window the analysis uses with a window of the same
length where no response is expected.

- Somatic: the spike counts against the commanded step (negative steps should give no spikes).
- Dendritic: the area of (G - G0)/R0 from 0.95 to 1.5 s (current injections at about 1 s) against
  the same length from 2.5 s.
- Sr2+-oEPSC: events from 40 to 400 ms after the flash, from the flash time in the optogenetic
  pulses table, against 500 to 860 ms.
- GRAB-ACh3.0: the mean fluorescence in the 1 s after the trial table's stimulus time against the 1 s
  before it.

In [ ]:
neg = som_sweeps[som_sweeps.step_pA < 0]
big = som_sweeps[som_sweeps.step_pA >= 200]
print(f"somatic: sweeps with negative steps that have spikes: {(neg.n_spikes > 0).sum()} of {len(neg)}; "
      f"sweeps at >= 200 pA with spikes: {(big.n_spikes > 0).sum()} of {len(big)}")
print(f"dendritic: proximal area in the analysis window median {den_pairs.prox_area.median():.3f}, late window {den_pairs.prox_late.median():.3f}; "
      f"distal {den_pairs.dist_area.median():.3f} against {den_pairs.dist_late.median():.3f}")
print("dendritic pairs with analysis-window area larger than the late window (proximal):",
      int((den_pairs.prox_area > den_pairs.prox_late).sum()), "of", len(den_pairs))
srr = sr_sweeps.groupby("path").agg(early=("n_events", "sum"), late=("n_events_late", "sum"))
print("Sr2+: events 40-400 ms against 500-860 ms after the flash, per file:")
print(srr.assign(path=srr.index.str.split("_ses-").str[1]).reset_index(drop=True).to_string())
g = grab_trials[grab_trials.stimulation == "single 1 ms pulse"]
def pre_post(r):
    s = r.stim - r.start
    pre = r.circle[(r.t >= s - 1) & (r.t < s)].mean()
    post = r.circle[(r.t >= s) & (r.t < s + 1)].mean()
    return post / pre - 1
gp = g.apply(pre_post, axis=1)
print(f"GRAB single-pulse trials: (mean after / mean before) - 1, median {gp.median():.3f}, "
      f"positive in {(gp > 0).sum()} of {len(gp)} trials")

In [ ]:

# Recordings deposited more than once: identical per-sweep signatures between two cells or regions
sig = som_sweeps.sort_values("start").groupby(["path", "cell", "condition"]).apply(
    lambda d: tuple(np.round(d.vmin, 3)) + tuple(np.round(d.vmax, 3)))
print("somatic cells:", len(sig), "; cells sharing their sweep minima and maxima with another cell:", int(sig.duplicated(keep=False).sum()))
dsig = den_pairs.groupby(["path", "cell", "condition", "proximal", "distal"]).apply(lambda d: (round(d.prox_area.iloc[0], 6), round(d.dist_area.iloc[0], 6)))
print("dendritic pairs:", len(dsig), "; pairs with the same areas as another pair:", int(dsig.duplicated(keep=False).sum()))
ssig = sr_sweeps.groupby(["path", "cell"]).apply(lambda d: tuple(np.round(d.noise_sd, 4)))
print("Sr2+ cells:", len(ssig), "; cells with the same noise in every sweep as another:", int(ssig.duplicated(keep=False).sum()))
gsig = grab_trials.groupby(["path", "region"]).apply(lambda d: tuple(np.round(np.concatenate([x[:5] for x in d.circle]), 2)))
print("GRAB regions:", len(gsig), "; regions whose trials start with the same frames as another region's:", int(gsig.duplicated(keep=False).sum()))
asig = aim[aim.behavior != "total"].sort_values(["session", "timestamp", "behavior"]).groupby("mouse").score.apply(tuple)
print("AIM mice:", len(asig), "; mice whose 5 sessions of scores equal another mouse's:", int(asig.duplicated(keep=False).sum()))


**Data check.** The NWB files hold the patch-clamp recordings of the off-state, on-state and
drug groups of Figures 1, 3, 6, 7 and 9, the Sr2+-oEPSC recordings of Figures 2 and 4, the
GRAB-ACh3.0 movies of Figure 5, the raw 2PLSM image stacks behind the spine counts, and the AIM
score sheets (with videos) of Figures 8 and 10. The output above shows the counts, units and clock
checks; each statement in the summary rests on these numbers. The somatic and dendritic recordings
of the control and 6-OHDA groups, the open-field data and the confocal images are not in the
dandiset, as its description says (cell `finding-missing-groups`).

In [ ]:
# What the dandiset does not hold, from the file list alone
ice_groups = assets[assets.kind.isin(["icephys", "icephys+ophys"])].group.unique().tolist()
print("groups among current-clamp files:", sorted(ice_groups))
print("files of unlesioned or lesioned control mice with patch-clamp data:",
      int(assets[assets.kind.str.contains("icephys") & assets.group.isin(["UnlesionedControl", "LesionedControl"])].shape[0]))
print("files that mention open field or confocal imaging in their name:", int(assets.path.str.contains("(?i)open|confocal").sum()))
print("spine-density files (raw image stacks only):", int((assets.nwb & (assets.kind == "") & assets.cell_type.isin(["dSPN", "iSPN"])).sum()))
print("spine-density files by cell type and group:", assets[assets.nwb & (assets.kind == "") & assets.cell_type.isin(["dSPN", "iSPN"])].groupby(["cell_type", "group"]).size().to_dict())

# What a spine file holds: one file per cell type and group, its top-level groups and their contents (metadata only)
def spine_contents(a):
    h = zhai.open_h5(a.asset_id)
    out = {}
    for top in ["acquisition", "processing", "analysis"]:
        kinds = {}
        def visit(name, o):
            nt = zhai._str(o.attrs.get("neurodata_type", "")) or ("dataset" if not hasattr(o, "keys") else "group")
            kinds[nt] = kinds.get(nt, 0) + 1
        h[top].visititems(visit)
        out[top] = kinds
    return dict(cell_type=a.cell_type, group=a.group, **out)
sp1 = assets[assets.nwb & (assets.kind == "") & assets.cell_type.isin(["dSPN", "iSPN"])].groupby(["cell_type", "group"]).head(1)
print(pd.DataFrame(zhai.pmap(spine_contents, [r for _, r in sp1.iterrows()])).to_string())

## Somatic excitability

### link-somatic-cells

> The dandiset's somatic current-step recordings are those behind the paper's somatic excitability
> figures, and the spike counts computed from them agree with the paper's per-cell tables.

Spike counts as in the paper's script: upward crossings of 0 mV over the sweep; rheobase the first
positive step with a spike. When a cell holds a step twice, the first sweep is used (an
assumption; the last is shown as a variant). Each dandiset cell is compared with every row of the
paper's table for its group over the steps from 20 to 300 pA present in both.

In [ ]:
SOM_MAP = [
    ("Fig 1E_dSPN somatic excitabilit", "dSPN", "LID off-state", "LID off"),
    ("Fig 1E_dSPN somatic excitabilit", "dSPN", "LID on-state", "LID on"),
    ("Fig 1E_dSPN somatic excitabilit", "dSPN", "LID on-state with SCH 23390", "LID on + SCH"),
    ("Fig 3C_iSPN somatic excitabilit", "iSPN", "LID off-state", "LID off"),
    ("Fig 3C_iSPN somatic excitabilit", "iSPN", "LID on-state", "LID on"),
    ("Fig 3C_iSPN somatic excitabilit", "iSPN", "LID on-state with sulpiride", "LID on + sul"),
    ("Fig 6C_M1R antagonist SE", "iSPN", "LID off-state, vehicle control", "control"),
    ("Fig 6C_M1R antagonist SE", "iSPN", "LID off-state with M1R antagonists", "M1R antagonist"),
    ("Fig 7C_KO somatic excitability", "iSPN", "CDGI knockout, LID off-state", "KO off"),
    ("Fig 7C_KO somatic excitability", "iSPN", "CDGI knockout, LID on-state", "KO on"),
    ("Fig 9D_M1R CRISPR SE", "iSPN", "LID off-state, control (gRNA only)", "control"),
    ("Fig 9D_M1R CRISPR SE", "iSPN", "LID off-state, M1R deletion", '"M1R"'),
]
RHEO_SHEET = {"Fig 1E_dSPN somatic excitabilit": "Fig 1F_dSPN rheobase", "Fig 3C_iSPN somatic excitabilit": "Fig 3D_iSPN rheobase"}

def somatic_link(cells):
    rows = []
    for sheet, ct, cond, pg in SOM_MAP:
        P = T[sheet]
        P = P[P.iloc[:, 1] == pg].reset_index(drop=True)
        stepcols = {int(str(c).split(" pA")[0]): c for c in P.columns[2:]}
        R = T[RHEO_SHEET[sheet]] if sheet in RHEO_SHEET else None
        if R is not None:
            R = R[R.iloc[:, 1] == pg].reset_index(drop=True)
        D = cells[(cells.cell_type == ct) & (cells.condition == cond)].sort_values(["date", "cell"])
        for _, d in D.iterrows():
            steps = [s for s in d.counts if s in stepcols and 20 <= s <= 300]
            diffs = [sum(abs(d.counts[s] - p[stepcols[s]]) for s in steps) for _, p in P.iterrows()]
            j = int(np.argmin(diffs))
            prow = P.iloc[j]
            pos = [s for s in sorted(stepcols) if s > 0 and prow[stepcols[s]] >= 1]
            fi_rheo = float(pos[0]) if pos else np.nan
            prheo = float(R.iloc[j, 2]) if R is not None else fi_rheo
            rows.append(dict(sheet=sheet[:6], group=pg, condition=cond, date=d.date, cell=d.cell, n_data=len(D),
                             n_paper=len(P), steps=len(steps), best_row=j, best_diff=diffs[j], n_tied=int(np.sum(np.array(diffs) == diffs[j])),
                             rheobase=d.rheobase, paper_rheobase=prheo, paper_fi_rheobase=fi_rheo))
    return pd.DataFrame(rows)

slink = somatic_link(som_cells)
summ = slink.groupby(["sheet", "group"], sort=False).agg(n_data=("n_data", "first"), n_paper=("n_paper", "first"),
        exact=("best_diff", lambda x: int((x == 0).sum())), within3=("best_diff", lambda x: int((x <= 3).sum())),
        rheobase_equal=("rheobase", "size"))
summ["rheobase_equal"] = slink.assign(eq=slink.rheobase == slink.paper_rheobase).groupby(["sheet", "group"], sort=False).eq.sum().values
print(summ.to_string())
print(f"\ngroups with the paper's n: {(summ.n_data == summ.n_paper).sum()} of {len(summ)}")
print(f"cells: {len(slink)}; exact match {(slink.best_diff == 0).sum()} ({(slink.best_diff == 0).mean():.1%}); "
      f"best match within 3 spikes {(slink.best_diff <= 3).sum()} ({(slink.best_diff <= 3).mean():.1%})")
print(f"rheobase equal to the best-matching row's: {(slink.rheobase == slink.paper_rheobase).sum()} of {len(slink)}")
print("\ncells whose best match differs by more than 3 spikes:")
print(slink[slink.best_diff > 3][["group", "condition", "date", "cell", "steps", "best_diff", "rheobase", "paper_rheobase"]].to_string())
reused = slink.groupby(["sheet", "group", "best_row"]).size()
print(f"\npaper rows that are the best match of more than one cell: {int((reused > 1).sum())}, "
      f"taking {int(reused[reused > 1].sum())} cells; cells whose best match is tied between rows: {int((slink.n_tied > 1).sum())}")
# The rheobase sheets (1F, 3D) are read by row number; check them against the F-I sheet's row
for sheet, rs in RHEO_SHEET.items():
    for pg in T[sheet].iloc[:, 1].unique():
        P = T[sheet][T[sheet].iloc[:, 1] == pg].reset_index(drop=True)
        R = T[rs][T[rs].iloc[:, 1] == pg].reset_index(drop=True)
        stepcols = {int(str(c).split(" pA")[0]): c for c in P.columns[2:]}
        fr = [next((s for s in sorted(stepcols) if s > 0 and P.loc[k, stepcols[s]] >= 1), np.nan) for k in range(len(P))]
        n = min(len(P), len(R))
        print(f"{rs}, {pg}: rows {len(R)} (F-I sheet {len(P)}); rheobase equal to the first step with a spike in the same row of the F-I sheet: "
              f"{int(np.sum(np.array(fr[:n], float) == R.iloc[:n, 2].to_numpy(float)))} of {n}")

print("\ncells with a repeated step:", int((som_cells.repeated_sweeps > 0).sum()), "of", len(som_cells))
alt = zhai.per_cell_somatic(som_sweeps, keep="last")
alt_link = somatic_link(alt)
print(f"variant, the last sweep of a repeated step: exact {(alt_link.best_diff == 0).sum()}, within 3 {(alt_link.best_diff <= 3).sum()} of {len(alt_link)}")
fl = som_cells[som_cells.repeated_sweeps > 0][["cell_type", "condition", "date", "cell", "repeated_sweeps", "rheobase", "fi_sum"]].merge(
    alt[["date", "cell", "condition", "path", "rheobase", "fi_sum"]].assign(path=None).drop(columns="path"),
    on=["date", "cell", "condition"], suffixes=("_first", "_last"))
fl = fl.merge(slink[["date", "cell", "condition", "best_diff"]], on=["date", "cell", "condition"]).merge(
    alt_link[["date", "cell", "condition", "best_diff"]], on=["date", "cell", "condition"], suffixes=("_first", "_last"))
print("cells with a repeated step, first against last sweep:"); print(fl.to_string())
print(f"cells whose rheobase changes: {int((fl.rheobase_first != fl.rheobase_last).sum())}; whose summed spikes change: {int((fl.fi_sum_first != fl.fi_sum_last).sum())}")

**Result: partially reproduced.** Every group holds the paper's number of cells (12 of 12 groups). Of 123 cells, 90 (73.2%) match a row of the paper's table exactly and 105 (85.4%) within 3 spikes, below the 80% and 95% the criterion asks for reproduced and above the 50% exact for partially reproduced. Rheobase equals that of the best-matching row in 105 cells. The best matches are not one to one: 9 paper rows are the best match of more than one cell (20 cells), and 6 cells tie between rows. The rheobase sheets (1F, 3D) are read by row number; in 115 of 124 rows their rheobase equals the first step with a spike in the same row of the F-I sheet, so the two sheets' rows mostly line up. Taking the last sweep of a repeated step changes none of the link counts: of the 14 cells with a repeated step, none changes rheobase and one changes its summed spikes, by 1.

In [ ]:
def by_group(df, ct, conds, col):
    return [df[(df.cell_type == ct) & (df.condition == c)][col].to_numpy(float) for c in conds]

def show_mw(label, a, b, la, lb):
    r = zhai.mw(a, b)
    print(f"{label}: {la} n={r['n1']} median {r['median1']:.3f}; {lb} n={r['n2']} median {r['median2']:.3f}; "
          f"Mann-Whitney U={r['U']:.1f}, P={r['p']:.4g}")
    return r

def per_date(df, ct, conds, col):
    d = df[(df.cell_type == ct) & df.condition.isin(conds)]
    return d.groupby(["condition", "date"])[col].agg(["size", "median"]).round(3)

def step_medians(cells, ct, conds):
    out = {}
    for c in conds:
        d = cells[(cells.cell_type == ct) & (cells.condition == c)]
        steps = sorted({s for x in d.counts for s in x})
        out[c] = {s: float(np.median([x[s] for x in d.counts if s in x])) for s in steps if 20 <= s <= 300}
    return pd.DataFrame(out).T

### fig1f-dspn-rheobase-on-lower

> In dSPNs the rheobase is lower in the LID on-state than in the off-state.

Two-sided Mann-Whitney test of per-cell rheobase, on-state against off-state.

In [ ]:
on, off = by_group(som_cells, "dSPN", ["LID on-state", "LID off-state"], "rheobase")
r_1f = show_mw("dSPN rheobase", on, off, "on", "off")
print("by recording day:"); print(per_date(som_cells, "dSPN", ["LID on-state", "LID off-state"], "rheobase").to_string())
alt_on, alt_off = by_group(alt, "dSPN", ["LID on-state", "LID off-state"], "rheobase")
show_mw("variant, last sweep of a repeated step", alt_on, alt_off, "on", "off");

**Result: reproduced.** On-state rheobase median 140 pA (n = 15) against off-state 220 pA (n = 15), Mann-Whitney P = 8.9e-06. The last-sweep variant gives the same, as no cell's rheobase changes with it.

### fig1e-dspn-fi-on-higher

> In on-state dSPNs suprathreshold current steps evoke more spikes than in off-state dSPNs across a
> range of intensities.

Per cell, the sum of spike counts from 20 to 300 pA over the steps present; two-sided Mann-Whitney.

In [ ]:
on, off = by_group(som_cells, "dSPN", ["LID on-state", "LID off-state"], "fi_sum")
show_mw("dSPN summed spikes 20-300 pA", on, off, "on", "off")
print("per-step medians (spikes):")
print(step_medians(som_cells, "dSPN", ["LID off-state", "LID on-state"]).to_string())
print("steps per cell from 20 to 300 pA:", som_cells[som_cells.cell_type == "dSPN"].groupby("condition").n_steps_20_300.agg(["min", "max"]).to_dict("index"))

**Result: reproduced.** Summed spikes from 20 to 300 pA, on-state median 88 against off-state 36, P = 8.8e-05; per-step medians are higher in the on-state from 120 pA up.

### fig1f-dspn-sch-no-change

> Bath application of the D1R antagonist SCH 23390 does not reverse the lower rheobase of on-state
> dSPNs.

A claim of absence: two-sided Mann-Whitney test of rheobase, on-state against on-state with SCH;
reproduced when P >= 0.05. Variant: Wilcoxon test on cells recorded in both conditions.

In [ ]:
on, sch = by_group(som_cells, "dSPN", ["LID on-state", "LID on-state with SCH 23390"], "rheobase")
show_mw("dSPN rheobase", on, sch, "on", "on + SCH")
d = som_cells[som_cells.cell_type == "dSPN"]
both = d[d.condition.isin(["LID on-state", "LID on-state with SCH 23390"])].pivot_table(index=["path", "cell"], columns="condition", values="rheobase").dropna()
print("cells recorded in both conditions:", len(both)); print(both.to_string())
if len(both) >= 2:
    print("Wilcoxon:", stats.wilcoxon(both.iloc[:, 0], both.iloc[:, 1]) if (both.iloc[:, 0] != both.iloc[:, 1]).any() else "all differences zero")

**Result: reproduced.** On-state median 140 pA (n = 15) against on-state with SCH 140 pA (n = 10), P = 1.0. The 5 cells recorded in both conditions give a Wilcoxon P = 1.0.

### fig3d-ispn-rheobase-on-higher

> In iSPNs the rheobase is higher in the on-state than in the off-state.

In [ ]:
on, off = by_group(som_cells, "iSPN", ["LID on-state", "LID off-state"], "rheobase")
show_mw("iSPN rheobase", on, off, "on", "off")
print(per_date(som_cells, "iSPN", ["LID on-state", "LID off-state"], "rheobase").to_string())
alt_on, alt_off = by_group(alt, "iSPN", ["LID on-state", "LID off-state"], "rheobase")
show_mw("variant, last sweep of a repeated step", alt_on, alt_off, "on", "off");

**Result: reproduced.** On-state median 160 pA (n = 11) against off-state 120 pA (n = 12), P = 0.00083.

### fig3c-ispn-fi-on-lower

> The F-I relationship of iSPNs shifts to the right in the on-state.

In [ ]:
on, off = by_group(som_cells, "iSPN", ["LID on-state", "LID off-state"], "fi_sum")
show_mw("iSPN summed spikes 20-300 pA", on, off, "on", "off")
print(step_medians(som_cells, "iSPN", ["LID off-state", "LID on-state"]).to_string())

**Result: reproduced.** Summed spikes, on-state median 65 against off-state 118.5, P = 7.2e-05.

### fig3d-ispn-sul-no-change

> Sulpiride does not alter the somatic excitability of on-state iSPNs.

A claim of absence: two-sided Mann-Whitney test of rheobase, on-state against on-state with
sulpiride; reproduced when P >= 0.05. Variant: Wilcoxon on cells recorded in both.

In [ ]:
on, sul = by_group(som_cells, "iSPN", ["LID on-state", "LID on-state with sulpiride"], "rheobase")
show_mw("iSPN rheobase", on, sul, "on", "on + sulpiride")
d = som_cells[som_cells.cell_type == "iSPN"]
both = d[d.condition.isin(["LID on-state", "LID on-state with sulpiride"])].pivot_table(index=["path", "cell"], columns="condition", values="rheobase").dropna()
print("cells recorded in both conditions:", len(both)); print(both.to_string())

**Result: reproduced.** On-state median 160 pA (n = 11) against sulpiride 120 pA (n = 7), P = 0.143, so the test finds no difference. Only 2 cells were recorded in both conditions, too few for the Wilcoxon variant.

### fig6c-m1r-antag-somatic-unchanged

> M1R antagonism throughout the off-state does not have a strong impact on the elevated somatic
> excitability of off-state iSPNs.

A claim of absence: two-sided Mann-Whitney test of rheobase, off-state vehicle control against M1R
antagonist; reproduced when P >= 0.05. Variant: summed spike counts.

In [ ]:
conds = ["LID off-state, vehicle control", "LID off-state with M1R antagonists"]
a, b = by_group(som_cells, "iSPN", conds, "rheobase")
show_mw("iSPN rheobase", a, b, "vehicle", "M1R antagonist")
a, b = by_group(som_cells, "iSPN", conds, "fi_sum")
show_mw("variant, summed spikes 20-300 pA", a, b, "vehicle", "M1R antagonist")
print(per_date(som_cells, "iSPN", conds, "rheobase").to_string())

**Result: reproduced.** Vehicle median 120 pA (n = 8) against M1R antagonist 140 pA (n = 10), P = 0.68; summed spikes P = 0.45.

### fig7c-cdgi-ko-somatic-oscillation

> Deletion of CDGI does not blunt the oscillation of iSPN somatic excitability between on- and
> off-states.

In CDGI KO iSPNs, two-sided Mann-Whitney test of rheobase, off-state against on-state; reproduced
when P < 0.05 with the on-state higher.

In [ ]:
conds = ["CDGI knockout, LID off-state", "CDGI knockout, LID on-state"]
off, on = by_group(som_cells, "iSPN", conds, "rheobase")
show_mw("CDGI KO iSPN rheobase", off, on, "off", "on")
print(per_date(som_cells, "iSPN", conds, "rheobase").to_string())

**Result: reproduced.** In CDGI KO iSPNs the on-state rheobase (median 210 pA, n = 8) is higher than the off-state (140 pA, n = 13), P = 0.0010.

### fig9d-m1r-crispr-somatic-unchanged

> Deletion of M1Rs from iSPNs does not alter the elevated somatic excitability of off-state iSPNs.

A claim of absence: two-sided Mann-Whitney test of rheobase, interleaved control (gRNA only)
against M1R CRISPR; reproduced when P >= 0.05.

In [ ]:
conds = ["LID off-state, control (gRNA only)", "LID off-state, M1R deletion"]
a, b = by_group(som_cells, "iSPN", conds, "rheobase")
show_mw("iSPN rheobase", a, b, "control", "M1R deletion")
a2, b2 = by_group(som_cells, "iSPN", conds, "fi_sum")
show_mw("variant, summed spikes 20-300 pA", a2, b2, "control", "M1R deletion")
print(per_date(som_cells, "iSPN", conds, "rheobase").to_string())

**Result: reproduced.** Control median 160 pA (n = 6) against M1R deletion 120 pA (n = 8), P = 0.147; summed spikes P = 0.24.

### Somatic tests under the last sweep of a repeated step

Every somatic test above, rheobase and summed spikes from 20 to 300 pA, with the first sweep of a
repeated step (main) and with the last sweep (variant, `alt` from the link cell).

In [ ]:
TESTS = [("dSPN", "LID on-state", "LID off-state"), ("dSPN", "LID on-state", "LID on-state with SCH 23390"),
         ("iSPN", "LID on-state", "LID off-state"), ("iSPN", "LID on-state", "LID on-state with sulpiride"),
         ("iSPN", "LID off-state, vehicle control", "LID off-state with M1R antagonists"),
         ("iSPN", "CDGI knockout, LID off-state", "CDGI knockout, LID on-state"),
         ("iSPN", "LID off-state, control (gRNA only)", "LID off-state, M1R deletion")]
rows = []
for ct, c1, c2 in TESTS:
    for col in ["rheobase", "fi_sum"]:
        r = {}
        for name, df in [("first", som_cells), ("last", alt)]:
            a, b = by_group(df, ct, [c1, c2], col)
            m = zhai.mw(a, b)
            r[f"U_{name}"], r[f"P_{name}"] = m["U"], m["p"]
        rows.append(dict(cell_type=ct, a=c1, b=c2, measure=col, **r))
lt = pd.DataFrame(rows)
print(lt.to_string(float_format=lambda x: f"{x:.4g}"))
print("tests whose P differs between first and last sweep:", int((lt.P_first != lt.P_last).sum()), "of", len(lt))
print("tests on opposite sides of 0.05:", int(((lt.P_first < 0.05) != (lt.P_last < 0.05)).sum()))


All 14 somatic tests give the same U and P with the last sweep of a repeated step as with the first.
The one cell whose summed spikes change (2020-02-17 cell 1, vehicle control, 170 to 171) does not move the
fig6c summed-spikes variant (P = 0.45 both ways).

## Dendritic excitability

### link-dendritic-cells

> The dandiset's dendritic line scans are those behind the paper's dendritic excitability index
> figures, and the index computed from them agrees with the paper's per-cell values.

Figure 6D names cells by date and letter (a, b, c for the first, second and third cell of the
day); elsewhere the paper's rows are taken to follow the dandiset's order by date and cell number.
The distance to the nearest paper value of the group is shown too, as a check of that order.

In [ ]:
DEN_MAP = [
    ("Fig 1I_dSPN dendritic excitabil", "dSPN", "LID off-state", "LID off"),
    ("Fig 1I_dSPN dendritic excitabil", "dSPN", "LID on-state", "LID on"),
    ("Fig 1I_dSPN dendritic excitabil", "dSPN", "LID on-state with SCH 23390", "LID on + SCH"),
    ("Fig 3F_iSPN dendritic excitabil", "iSPN", "LID off-state", "LID off"),
    ("Fig 3F_iSPN dendritic excitabil", "iSPN", "LID on-state", "LID on"),
    ("Fig 3F_iSPN dendritic excitabil", "iSPN", "LID on-state with sulpiride", "LID on + sul"),
    ("Fig 6D_M1R antagonist DE", "iSPN", "LID off-state, vehicle control", "control"),
    ("Fig 6D_M1R antagonist DE", "iSPN", "LID off-state with M1R antagonists", "M1R antagonist"),
    ("Fig 7F_KO dendritic excitabilit", "iSPN", "CDGI knockout, LID off-state", ("KO", "LID off")),
    ("Fig 7F_KO dendritic excitabilit", "iSPN", "CDGI knockout, LID on-state", ("KO", "LID on")),
]

def dendritic_link(cells, by_ordinal=False):
    rows = []
    for sheet, ct, cond, pg in DEN_MAP:
        P = T[sheet]
        P = P[(P.iloc[:, 1] == pg[0]) & (P.iloc[:, 2] == pg[1])] if isinstance(pg, tuple) else P[P.iloc[:, 1] == pg]
        vals = P.iloc[:, -1].to_numpy(float)
        ids = P.iloc[:, 0].astype(str).tolist()
        D = cells[(cells.cell_type == ct) & (cells.condition == cond)].sort_values(["date", "cell"]).reset_index(drop=True)
        D["ordinal"] = D.groupby("date").cumcount()  # the cell's place among the group's analysed cells of the day
        for i, d in D.iterrows():
            if sheet.startswith("Fig 6D"):
                letter = "abcdefgh"[int(d.ordinal)] if by_ordinal else "abcdefgh"[int(d.cell) - 1]
                pid = f"{d.date[4:8]}{d.date[:4]}_{letter}"
                pv = vals[ids.index(pid)] if pid in ids else np.nan
            else:
                pid = ids[i] if i < len(ids) else None
                pv = vals[i] if i < len(vals) else np.nan
            rows.append(dict(sheet=sheet[:6], group=str(pg), date=d.date, cell=d.cell, n_pairs=d.n_pairs,
                             n_data=len(D), n_paper=len(P), paper_id=pid, ordinal=int(d.ordinal) + 1, index=d["index"], paper=pv,
                             nearest=vals[np.argmin(np.abs(vals - d["index"]))]))
    L = pd.DataFrame(rows)
    L["rel_diff"] = (L["index"] - L.paper).abs() / L.paper
    L["rel_diff_nearest"] = (L["index"] - L.nearest).abs() / L.nearest
    return L

dlink = dendritic_link(den_cells)
dsum = dlink.groupby(["sheet", "group"], sort=False).agg(n_data=("n_data", "first"), n_paper=("n_paper", "first"),
        median_rel_diff=("rel_diff", "median"), within_1pct=("rel_diff", lambda x: int((x < 0.01).sum())),
        within_10pct=("rel_diff", lambda x: int((x <= 0.10).sum())))
print(dsum.round(3).to_string())
print(f"\ngroups with the paper's n: {(dsum.n_data == dsum.n_paper).sum()} of {len(dsum)}")
print(f"median relative difference from the matched paper value: {dlink.rel_diff.median():.3f} over {len(dlink)} cells; "
      f"to the nearest value of the group: {dlink.rel_diff_nearest.median():.3f}")
d6 = dlink[dlink.sheet == "Fig 6D"]
print(f"Figure 6D, letter from the file's cell number: {int(d6.paper.notna().sum())} of {len(d6)} cells find their identifier; "
      f"median relative difference {d6.rel_diff.median():.3f}; cells whose cell number differs from their place among the group's cells of the day: {int((d6.ordinal != d6.cell).sum())}")
v6 = dendritic_link(den_cells, by_ordinal=True)
v6 = v6[v6.sheet == "Fig 6D"]
print(f"variant, letter from the place among the group's cells of the day: {int(v6.paper.notna().sum())} of {len(v6)} find their identifier; "
      f"median relative difference {v6.rel_diff.median():.3f}")
print("paper's Figure 6D identifiers:", T["Fig 6D_M1R antagonist DE"].iloc[:, 0].tolist())
print("\ncells more than 10% from their matched value:")
print(dlink[dlink.rel_diff > 0.10][["group", "date", "cell", "n_pairs", "paper_id", "index", "paper", "nearest"]].round(3).to_string())

**Result: reproduced.** All 10 groups hold the paper's number of cells, and the index differs from the matched paper value by a median of 2.7% (1.7% to the nearest value of the group). Twelve cells differ by more than 10%; one of them, 2019-04-11 cell 1 (on-state with SCH), is 0.584 against the paper's 0.228 because only its second pair of sites is in the file (cell `finding-sch-cell-pair`). In Figure 6D the paper's letters follow the files' cell numbers: all 16 cells find their identifier that way, and 14 of 16 when the letter is taken as the cell's place among the group's cells of the day, because the paper names 02182020_b, 02182020_c and 02202020_c with no _a in the group.

In [ ]:
# 2019-04-11, cell 1 (on-state with SCH 23390): the pairs in the file and the paper's value
p = den_pairs[(den_pairs.date == "20190411") & (den_pairs.cell == 1)]
print(p[["condition", "proximal", "distal", "prox_area", "dist_area", "basal_ratio", "index"]].round(3).to_string())
print("paper (Fig 1I, LID on + SCH, row matched by order):", dlink[(dlink.date == "20190411") & (dlink.cell == 1)][["paper_id", "paper"]].to_dict("records"))

### fig1i-dspn-dendritic-on-higher

> The dendritic excitability index of dSPNs is higher in the on-state than in the off-state.

In [ ]:
on, off = by_group(den_cells, "dSPN", ["LID on-state", "LID off-state"], "index")
show_mw("dSPN dendritic index", on, off, "on", "off")
print(per_date(den_cells, "dSPN", ["LID on-state", "LID off-state"], "index").to_string())
print("paper's table, same groups:")
P = T["Fig 1I_dSPN dendritic excitabil"]; show_mw("paper table", P[P.Group == "LID on"].iloc[:, 2], P[P.Group == "LID off"].iloc[:, 2], "on", "off");

**Result: reproduced.** On-state index median 0.476 against off-state 0.329 (10 cells each), P = 0.0058; the paper's table gives P = 0.014.

### fig1i-dspn-dendritic-sch-lower

> SCH 23390 reverses the higher dendritic excitability of on-state dSPNs.

Two-sided Mann-Whitney test, on-state against on-state with SCH; reproduced when P < 0.05 and the
SCH median is lower. Variants: without the cell of 2019-04-11 whose first pair is missing, and the
paper's own table.

In [ ]:
on, sch = by_group(den_cells, "dSPN", ["LID on-state", "LID on-state with SCH 23390"], "index")
show_mw("dSPN dendritic index", on, sch, "on", "on + SCH")
keep = ~((den_cells.date == "20190411") & (den_cells.cell == 1))
on2, sch2 = by_group(den_cells[keep], "dSPN", ["LID on-state", "LID on-state with SCH 23390"], "index")
show_mw("variant, without 2019-04-11 cell 1", on2, sch2, "on", "on + SCH")
P = T["Fig 1I_dSPN dendritic excitabil"]
show_mw("paper table", P[P.Group == "LID on"].iloc[:, 2], P[P.Group == "LID on + SCH"].iloc[:, 2], "on", "on + SCH")
print(per_date(den_cells, "dSPN", ["LID on-state with SCH 23390"], "index").to_string())

**Result: not reproduced.** The SCH median is lower (0.419, n = 9, against 0.476, n = 10) but P = 0.066. Without 2019-04-11 cell 1, whose first pair of sites is missing from the file, P = 0.027 (n = 8); the paper's table gives P = 0.016. The exclusion is shown to locate the difference and does not decide the outcome: the paper analysed this cell, with a pair the file does not hold.

### fig3f-ispn-dendritic-on-lower

> The dendritic excitability index of iSPNs falls in the on-state compared with the off-state.

In [ ]:
on, off = by_group(den_cells, "iSPN", ["LID on-state", "LID off-state"], "index")
show_mw("iSPN dendritic index", on, off, "on", "off")
print(per_date(den_cells, "iSPN", ["LID on-state", "LID off-state"], "index").to_string())

**Result: reproduced.** On-state median 0.339 against off-state 0.524 (10 cells each), P = 0.0073.

### fig3f-ispn-dendritic-sul-no-change

> Sulpiride does not significantly alter dendritic excitability in on-state iSPNs.

A claim of absence: two-sided Mann-Whitney test; reproduced when P >= 0.05.

In [ ]:
on, sul = by_group(den_cells, "iSPN", ["LID on-state", "LID on-state with sulpiride"], "index")
show_mw("iSPN dendritic index", on, sul, "on", "on + sulpiride");

**Result: reproduced.** On-state median 0.339 (n = 10) against sulpiride 0.497 (n = 8), P = 0.0545, the paper's value; the test does not reach P < 0.05, by a small margin.

### fig6d-m1r-antag-dendritic-lower

> M1R antagonism prevents the off-state elevation of iSPN dendritic excitability.

In [ ]:
conds = ["LID off-state, vehicle control", "LID off-state with M1R antagonists"]
a, b = by_group(den_cells, "iSPN", conds, "index")
show_mw("iSPN dendritic index", a, b, "vehicle", "M1R antagonist")
print(per_date(den_cells, "iSPN", conds, "index").to_string())

**Result: reproduced.** Vehicle median 0.451 (n = 9) against M1R antagonist 0.269 (n = 7), P = 0.023.

### fig7f-cdgi-ko-dendritic-unchanged

> The off-state elevation of iSPN dendritic excitability is absent in CDGI KO mice (off against on
> not different), while it is present in wild type.

A claim of absence for the KO: two-sided Mann-Whitney test, KO off-state against KO on-state;
reproduced when P >= 0.05. The wild-type part is fig3f-ispn-dendritic-on-lower.

In [ ]:
conds = ["CDGI knockout, LID off-state", "CDGI knockout, LID on-state"]
a, b = by_group(den_cells, "iSPN", conds, "index")
show_mw("CDGI KO iSPN dendritic index", a, b, "off", "on")
on, off = by_group(den_cells, "iSPN", ["LID on-state", "LID off-state"], "index")
show_mw("wild type (fig3f)", off, on, "off", "on");

**Result: reproduced.** KO off-state median 0.312 (n = 11) against KO on-state 0.286 (n = 8), P = 0.31; in wild type off-state 0.524 against on-state 0.339, P = 0.0073.

## Sr2+-oEPSCs

### link-sr-cells

> The dandiset's Sr2+-oEPSC recordings are cells of the paper's Figures 2 and 4, and their events
> agree with the paper's per-cell values.

Events are found with the detector in `zhai.detect_events` (a stand-in for TaroTools; see the
helper's docstring): from 40 to 400 ms after the flash, peaks of inward current whose prominence
exceeds five SD of the noise. The paper's identifiers are the date followed by a letter (the
dSPN table writes the date as MMDDYYYY, the iSPN table as YYYYMMDD); the letter is the cell
number of the file. The paper's event count per cell is its frequency times 3.6 s (10 sweeps of
0.36 s), which is what its frequencies imply.

In [ ]:
def sr_cells(sw, amp_col="amps", n_col="n_events", max_sweeps=None, last=False):
    if max_sweeps:
        g = sw.sort_values("sweep_start").groupby(["path", "cell"])
        sw = g.tail(max_sweeps) if last else g.head(max_sweeps)
    c = sw.groupby(["cell_type", "date", "cell", "condition"]).agg(n_sweeps=(n_col, "size"), n_events=(n_col, "sum"),
                                                                      amps=(amp_col, "sum")).reset_index()
    c["mean_amp"] = c.amps.apply(lambda x: float(np.mean(x)) if len(x) else np.nan)
    c["freq_hz"] = c.n_events / (c.n_sweeps * 0.36)
    c["key"] = c.date + c.cell.apply(lambda k: "abcdefgh"[k - 1])
    return c.drop(columns="amps")

def paper_sr():
    P = []
    for ct, a, f in [("dSPN", "Fig 2H_dSPN Sr-oEPSC amp", "Fig 2I_dSPN Sr-oEPSC freq"), ("iSPN", "Fig 4G_iSPN Sr-oEPSC amp", "Fig 4H_iSPN Sr-oEPSC freq")]:
        x = T[a].set_axis(["id", "group", "paper_amp"], axis=1).merge(T[f].set_axis(["id", "group", "paper_freq"], axis=1))
        x["ct"] = ct
        P.append(x)
    P = pd.concat(P, ignore_index=True)
    P["id"] = P.id.astype(str)
    P["key"] = [i[4:8] + i[:4] + i[8:] if ct == "dSPN" else i for i, ct in zip(P.id, P.ct)]
    P["paper_n"] = P.paper_freq * 3.6
    return P

def sr_link(c):
    M = c.merge(paper_sr(), on="key", how="outer", indicator=True)
    M["amp_rel"] = (M.mean_amp - M.paper_amp) / M.paper_amp
    M["n_rel"] = (M.n_events - M.paper_n) / M.paper_n
    M["within25"] = (M.amp_rel.abs() <= 0.25) & (M.n_rel.abs() <= 0.25)
    return M

src = sr_cells(sr_sweeps)
M = sr_link(src)
print(M[["key", "cell_type", "condition", "group", "n_sweeps", "n_events", "paper_n", "mean_amp", "paper_amp", "amp_rel", "n_rel", "_merge"]].round(2).to_string())
print(f"\ndandiset cells {len(src)}, paper cells {M.id.notna().sum()}, matched {(M._merge == 'both').sum()}, "
      f"dandiset cells without a paper identifier {(M._merge == 'left_only').sum()}, paper cells not in the dandiset {(M._merge == 'right_only').sum()}")
print(f"cells with mean amplitude and event count both within 25% of the paper's: {M.within25.sum()} of {len(M)} ({M.within25.mean():.0%})")
print(f"amplitude within 25%: {(M.amp_rel.abs() <= 0.25).sum()}; count within 25%: {(M.n_rel.abs() <= 0.25).sum()}")
print(f"median relative difference: amplitude {M.amp_rel.median():+.3f}, count {M.n_rel.median():+.3f}")
for label, kw in [("variant, noise SD of the smoothed trace", dict(amp_col="amps_smoothed_sd", n_col="n_events_smoothed_sd")),
                  ("variant, first 10 sweeps of each cell", dict(max_sweeps=10)),
                  ("variant, last 10 sweeps of each cell", dict(max_sweeps=10, last=True))]:
    Mv = sr_link(sr_cells(sr_sweeps, **kw))
    print(f"{label}: both within 25% for {Mv.within25.sum()} of {len(Mv)}; median amplitude difference {Mv.amp_rel.median():+.3f}, count {Mv.n_rel.median():+.3f}")

**Result: partially reproduced.** All 33 dandiset cells match a paper identifier and all 33 paper cells are in the dandiset. Mean amplitude and event count are both within 25% of the paper's for 21 of 33 cells (64%), below the 75% the criterion asks; amplitude alone for 25 and count alone for 26. Median differences are +11% in amplitude and -2% in count. With the first 10 sweeps 21 cells are within 25%, with the last 10 sweeps 22. The detector is a stand-in for TaroTools with visual checking; with the noise SD of the smoothed trace only 8 cells are within 25%.

In [ ]:
# Sweeps per Sr2+-oEPSC cell; the paper's frequencies imply 10
print(src[src.n_sweeps != 10][["cell_type", "date", "cell", "condition", "n_sweeps"]].to_string())
# For these cells, events in each run of 10 consecutive sweeps against the paper's count (frequency x 3.6 s)
Psr = paper_sr()
for _, c in src[src.n_sweeps > 10].iterrows():
    sw = sr_sweeps[(sr_sweeps.date == c.date) & (sr_sweeps.cell == c.cell)].sort_values("sweep_start")
    n = sw.n_events.to_numpy()
    runs = [int(n[k:k + 10].sum()) for k in range(len(n) - 9)]
    print(f"{c.date} cell {c.cell}: events per sweep {n.tolist()}; sweeps 1-10 onward, events in 10 sweeps {runs}; "
          f"paper {float(Psr[Psr.key == c.key].paper_n.iloc[0]):.1f}")
# A variant: for these cells, the 10 consecutive sweeps whose event count is nearest the paper's.
# Both runs of 2023-04-06 cell 1 are far above the paper's count, so this choice is weakly supported
parts = []
for (pth, cl), d in sr_sweeps.sort_values("sweep_start").groupby(["path", "cell"]):
    if len(d) > 10:
        n = d.n_events.to_numpy()
        pn = float(Psr[Psr.key == d.date.iloc[0] + "abcdefgh"[cl - 1]].paper_n.iloc[0])
        k = int(np.argmin([abs(n[k:k + 10].sum() - pn) for k in range(len(n) - 9)]))
        print(f"{d.date.iloc[0]} cell {cl}: nearest run starts at sweep {k + 1}")
        d = d.iloc[k:k + 10]
    parts.append(d)
sr_sweeps_nearest = pd.concat(parts)

### fig2h-dspn-sr-amplitude-on-larger

> The amplitude of Sr2+-oEPSCs evoked from cortical afferents is larger in on-state than in off-state dSPNs.

Per-cell mean event amplitude, on-state against
off-state; the claims file gives no criterion, so the paper's test is used: two-sided Mann-Whitney,
P < 0.05 for a difference, with the on-state larger. Variants: the smoothed-noise detector, the first 10 and the last 10 sweeps of
each cell, and the paper's own per-cell table.

In [ ]:
c = src[src.cell_type == "dSPN"]
show_mw("dSPN mean_amp", c[c.condition == "LID on-state"].mean_amp, c[c.condition == "LID off-state"].mean_amp, "on", "off")
for label, kw in [("variant, noise SD of the smoothed trace", dict(amp_col="amps_smoothed_sd", n_col="n_events_smoothed_sd")),
                  ("variant, first 10 sweeps", dict(max_sweeps=10)), ("variant, last 10 sweeps", dict(max_sweeps=10, last=True))]:
    v = sr_cells(sr_sweeps, **kw); v = v[v.cell_type == "dSPN"]
    show_mw(label, v[v.condition == "LID on-state"].mean_amp, v[v.condition == "LID off-state"].mean_amp, "on", "off")
P = paper_sr(); P = P[P.ct == "dSPN"]
col = "paper_amp" if "mean_amp" == "mean_amp" else "paper_freq"
show_mw("paper's table", P[P.group == "LID on"][col], P[P.group == "LID off"][col], "on", "off")
print(c.groupby(["condition", "date"])["mean_amp"].agg(["size", "median"]).round(2).to_string())

**Result: reproduced (resting on the stand-in detector).** On-state mean amplitude median 18.75 pA (n = 8) against off-state 13.44 pA (n = 10), P = 0.0062; the smoothed-noise detector gives P = 0.0044, the paper's table P = 0.00032.

### fig2i-dspn-sr-frequency-unchanged

> The frequency of Sr2+-oEPSCs in dSPNs does not differ between states.

Per-cell event frequency, on-state against
off-state; the claims file gives no criterion, so the paper's test is used: two-sided Mann-Whitney,
P < 0.05 for a difference; a claim of absence, reproduced when P >= 0.05. Variants: the smoothed-noise detector, the first 10 and the last 10 sweeps of
each cell, and the paper's own per-cell table.

In [ ]:
c = src[src.cell_type == "dSPN"]
show_mw("dSPN freq_hz", c[c.condition == "LID on-state"].freq_hz, c[c.condition == "LID off-state"].freq_hz, "on", "off")
for label, kw in [("variant, noise SD of the smoothed trace", dict(amp_col="amps_smoothed_sd", n_col="n_events_smoothed_sd")),
                  ("variant, first 10 sweeps", dict(max_sweeps=10)), ("variant, last 10 sweeps", dict(max_sweeps=10, last=True))]:
    v = sr_cells(sr_sweeps, **kw); v = v[v.cell_type == "dSPN"]
    show_mw(label, v[v.condition == "LID on-state"].freq_hz, v[v.condition == "LID off-state"].freq_hz, "on", "off")
P = paper_sr(); P = P[P.ct == "dSPN"]
col = "paper_amp" if "freq_hz" == "mean_amp" else "paper_freq"
show_mw("paper's table", P[P.group == "LID on"][col], P[P.group == "LID off"][col], "on", "off")
print(c.groupby(["condition", "date"])["freq_hz"].agg(["size", "median"]).round(2).to_string())

**Result: reproduced (resting on the stand-in detector).** Frequency medians 21.3 Hz on and 16.1 Hz off, P = 0.0999; the smoothed-noise detector P = 0.12; the paper's table P = 0.13.

### fig4g-ispn-sr-amplitude-on-smaller

> The amplitude of Sr2+-oEPSCs is smaller in on-state than in off-state iSPNs.

Per-cell mean event amplitude, on-state against
off-state; the claims file gives no criterion, so the paper's test is used: two-sided Mann-Whitney,
P < 0.05 for a difference, with the on-state smaller. Variants: the smoothed-noise detector, the first 10 and the last 10 sweeps of
each cell, the 10 consecutive sweeps nearest the paper's count, and the paper's own per-cell table.

In [ ]:
c = src[src.cell_type == "iSPN"]
show_mw("iSPN mean_amp", c[c.condition == "LID on-state"].mean_amp, c[c.condition == "LID off-state"].mean_amp, "on", "off")
for label, kw in [("variant, noise SD of the smoothed trace", dict(amp_col="amps_smoothed_sd", n_col="n_events_smoothed_sd")),
                  ("variant, first 10 sweeps", dict(max_sweeps=10)), ("variant, last 10 sweeps", dict(max_sweeps=10, last=True))]:
    v = sr_cells(sr_sweeps, **kw); v = v[v.cell_type == "iSPN"]
    show_mw(label, v[v.condition == "LID on-state"].mean_amp, v[v.condition == "LID off-state"].mean_amp, "on", "off")
v = sr_cells(sr_sweeps_nearest); v = v[v.cell_type == "iSPN"]
show_mw("variant, the 10 sweeps nearest the paper's count", v[v.condition == "LID on-state"].mean_amp, v[v.condition == "LID off-state"].mean_amp, "on", "off")
P = paper_sr(); P = P[P.ct == "iSPN"]
col = "paper_amp" if "mean_amp" == "mean_amp" else "paper_freq"
show_mw("paper's table", P[P.group == "LID on"][col], P[P.group == "LID off"][col], "on", "off")
print(c.groupby(["condition", "date"])["mean_amp"].agg(["size", "median"]).round(2).to_string())

**Result: reproduced (resting on the stand-in detector).** On-state median 9.71 pA (n = 8) against off-state 13.73 pA (n = 7), P = 0.0093; smoothed-noise detector P = 0.0037; the paper's table P = 0.00031.

### fig4h-ispn-sr-frequency-unchanged

> The frequency of Sr2+-oEPSCs in iSPNs does not differ between states.

Per-cell event frequency, on-state against
off-state; the claims file gives no criterion, so the paper's test is used: two-sided Mann-Whitney,
P < 0.05 for a difference; a claim of absence, reproduced when P >= 0.05. Variants: the smoothed-noise detector, the first 10 and the last 10 sweeps of
each cell, the 10 consecutive sweeps nearest the paper's count, and the paper's own per-cell table.

In [ ]:
c = src[src.cell_type == "iSPN"]
show_mw("iSPN freq_hz", c[c.condition == "LID on-state"].freq_hz, c[c.condition == "LID off-state"].freq_hz, "on", "off")
for label, kw in [("variant, noise SD of the smoothed trace", dict(amp_col="amps_smoothed_sd", n_col="n_events_smoothed_sd")),
                  ("variant, first 10 sweeps", dict(max_sweeps=10)), ("variant, last 10 sweeps", dict(max_sweeps=10, last=True))]:
    v = sr_cells(sr_sweeps, **kw); v = v[v.cell_type == "iSPN"]
    show_mw(label, v[v.condition == "LID on-state"].freq_hz, v[v.condition == "LID off-state"].freq_hz, "on", "off")
v = sr_cells(sr_sweeps_nearest); v = v[v.cell_type == "iSPN"]
show_mw("variant, the 10 sweeps nearest the paper's count", v[v.condition == "LID on-state"].freq_hz, v[v.condition == "LID off-state"].freq_hz, "on", "off")
P = paper_sr(); P = P[P.ct == "iSPN"]
col = "paper_amp" if "freq_hz" == "mean_amp" else "paper_freq"
show_mw("paper's table", P[P.group == "LID on"][col], P[P.group == "LID off"][col], "on", "off")
print(c.groupby(["condition", "date"])["freq_hz"].agg(["size", "median"]).round(2).to_string())

**Result: reproduced (resting on the stand-in detector and on which sweeps are used).** Frequency medians 14.9 Hz on and 24.2 Hz off, P = 0.064 with every sweep. Two off-state cells hold 12 and 11 sweeps where the paper's frequencies imply 10. Taking the last 10 sweeps, or the run of 10 consecutive sweeps whose event count is nearest the paper's for each cell (sweeps 1 to 10 and 2 to 11), also gives P = 0.064; taking the first 10 gives P = 0.049, which would find a difference. The readings differ only in 2023-04-06 cell 1, where sweeps 1 to 10 hold 93 events and sweeps 2 to 11 hold 89, both well above the paper's 73 (cell `finding-sr-sweeps`). The paper's per-cell counts therefore give little support for one run over the other: the choice rests on a difference of 4 events from a stand-in detector. The outcome rests on the assumption that the paper did not use the first 10 sweeps of that cell; it is fragile. The smoothed-noise detector gives P = 0.118 and the paper's table P = 0.23.

## GRAB-ACh3.0

### link-grab-rois

> The dandiset's GRAB-ACh recordings are regions of the paper's Figure 5F, and the normalised areas
> computed from them agree with the paper's per-region values.

Normalised areas as in the paper's script (see `zhai.grab_areas`), from the 50 um circle in the
centre of each movie. The criterion speaks of each mouse's vector; the files hold several
regions per mouse and the paper's n counts regions, so each region is matched (to the paper's
region in the same group with the smallest summed absolute difference over the treatments both
hold).

In [ ]:
DRUG = {"control": "No drug", "10 µM quinpirole": "quinpirole", "10 µM sulpiride": "sulpiride", "50 nM dopamine": "DA"}
GROUP = {"UnlesionedControl": "unlesioned", "LesionedControl": "6-OHDA", "OffState": "LID off"}

def grab_table(roi="circle", fmin_rule="smaller", max_trials=None):
    G = zhai.grab_areas(grab_trials, roi=roi, fmin_rule=fmin_rule, max_trials=max_trials)
    G = G[G.drug.isin(DRUG)]
    G["drug"] = G.drug.map(DRUG)
    G["group"] = G.group.map(GROUP)
    return G.pivot_table(index=["group", "date", "region"], columns="drug", values="auc").reset_index()

def grab_link(W):
    P = T["Fig 5F_GRAB-ACh_single-AUC"].set_axis(["roi", "group", "drug", "auc"], axis=1)
    PW = P.pivot_table(index=["group", "roi"], columns="drug", values="auc").reset_index()
    rows = []
    for _, w in W.iterrows():
        cand = PW[PW.group == w.group]
        drugs = [d for d in ["No drug", "quinpirole", "sulpiride", "DA"] if not np.isnan(w.get(d, np.nan))]
        diffs = [(np.nansum([abs(w[d] - p[d]) for d in drugs if not np.isnan(p[d])]), p) for _, p in cand.iterrows()]
        best, p = min(diffs, key=lambda x: x[0])
        maxd = max(abs(w[d] - p[d]) for d in drugs if not np.isnan(p[d]))
        rows.append(dict(group=w.group, date=w.date, region=w.region, paper_roi=p.roi, summed_diff=best, max_diff=maxd,
                         **{f"{d}": w[d] for d in drugs}, **{f"paper {d}": p[d] for d in drugs}))
    return pd.DataFrame(rows), PW

W = grab_table()
GL, PW = grab_link(W)
print(GL.round(3).to_string())
cnt = pd.DataFrame({"dandiset": W.groupby("group").size(), "paper": PW.groupby("group").size()})
print("\nregions per group:"); print(cnt.to_string())
print(f"regions whose best match is within 0.05 for every treatment: {(GL.max_diff <= 0.05).sum()} of {len(GL)}")
print("paper regions matched by more than one dandiset region:", int(GL.duplicated(["group", "paper_roi"]).sum()))
# The criterion's unit is the mouse; a mouse (recording day) matches when all its regions do
bym = GL.groupby(["group", "date"]).max_diff.max()
print(f"mice (recording days) whose every region is within 0.05: {int((bym <= 0.05).sum())} of {len(bym)}")
# One-to-one assignment within each group (each paper region used once), minimising the summed difference
from scipy.optimize import linear_sum_assignment
one = []
for grp, w in W.groupby("group"):
    cand = PW[PW.group == grp].reset_index(drop=True)
    drugs = ["No drug", "quinpirole", "sulpiride", "DA"]
    cost = np.array([[np.nansum([abs(r[d] - p[d]) for d in drugs if d in r and not np.isnan(r[d]) and not np.isnan(p[d])])
                      for _, p in cand.iterrows()] for _, r in w.iterrows()])
    ri, ci = linear_sum_assignment(cost)
    for a, b in zip(ri, ci):
        r, p = w.iloc[a], cand.iloc[b]
        one.append(max(abs(r[d] - p[d]) for d in drugs if d in r and not np.isnan(r[d]) and not np.isnan(p[d])))
one = np.array(one)
print(f"variant, one-to-one assignment: regions within 0.05 for every treatment {int((one <= 0.05).sum())} of {len(one)}")
for label, kw in [("whole-frame brightness", dict(roi="whole")), ("Fmin from tetrodotoxin only", dict(fmin_rule="ttx")), ("Fmin from the quinpirole baseline only", dict(fmin_rule="quin")), ("first two trials of each condition", dict(max_trials=2))]:
    v, _ = grab_link(grab_table(**kw))
    print(f"variant, {label}: within 0.05 for {(v.max_diff <= 0.05).sum()} of {len(v)}; median largest difference {v.max_diff.median():.3f}")

**Result: not reproduced.** The dandiset holds the paper's number of regions in each group (13 unlesioned, 10 6-OHDA, 11 off-state), but only 16 of 34 regions have a paper region within 0.05 for every treatment (47%, under the half the criterion asks for partially reproduced), and 11 paper regions are the best match of more than one dandiset region. Assigning paper regions one to one gives 15 of 34. Counted by mouse (recording day), as the criterion words it, 1 of 11 mice has every region within 0.05. The variants (whole-frame brightness, Fmin rules, two trials) give 10 to 16 of 34.

In [ ]:
# Calibration trials per region: acetylcholine (Fmax) and tetrodotoxin (Fmin), with their lengths
rows = []
for (grp, date, region), d in grab_trials.groupby(["group", "date", "region"]):
    ach = d[d.drug.str.contains("acetylcholine")]
    ttx = d[d.drug.str.contains("tetrodotoxin")]
    rows.append(dict(group=grp, date=date, region=region, n_ach=len(ach), ach_lengths_s=[round(float(r.t[-1]), 1) for _, r in ach.iterrows()],
                     n_ttx=len(ttx), single_pulse_trials=d[d.stimulation == "single 1 ms pulse"].groupby("drug").size().to_dict()))
cal = pd.DataFrame(rows)
print("regions:", len(cal))
print("regions without a tetrodotoxin trial, or with an acetylcholine trial shorter than the 2 to 4 s window:")
print(cal[(cal.n_ttx == 0) | cal.ach_lengths_s.apply(lambda x: len(x) == 0 or min(x) < 4)][["group", "date", "region", "n_ach", "ach_lengths_s", "n_ttx"]].to_string())
print("regions with more than two single-pulse trials in a condition:")
print(cal[cal.single_pulse_trials.apply(lambda x: max(x.values()) > 2)][["group", "date", "region", "single_pulse_trials"]].to_string())
# Frame interval within each single-pulse condition (trials are aligned with one interval per condition)
fdt = grab_trials[grab_trials.stimulation == "single 1 ms pulse"].groupby(["path", "region", "drug"]).frame_dt.agg(["min", "max"])
print(f"single-pulse conditions: {len(fdt)}; with frame intervals differing by more than 1%: {int((fdt['max'] > 1.01 * fdt['min']).sum())}; "
      f"intervals {fdt['min'].min():.4f} to {fdt['max'].max():.4f} s")
print("frame intervals over all trials (s):", grab_trials.frame_dt.round(3).value_counts().to_dict())
# 2024-04-02 Slice1Roi2: the description says it has no acetylcholine trial
s = grab_trials[(grab_trials.date == "20240402") & (grab_trials.region == "Slice1Roi2")]
print(s[["region", "drug", "stimulation", "start", "n_frames"]].assign(length_s=s.t.apply(lambda t: round(float(t[-1]), 2))).to_string())

### fig5f-ach-off-elevated

> Evoked ACh release (GRAB-ACh3.0) in slices from LID off-state mice is greater than in unlesioned striatum.

Single-pulse normalised area without drug, one value per region; two-sided Mann-Whitney, unlesioned against off-state.

In [ ]:
def nodrug(W, g): return W[W.group == g]["No drug"].dropna()
show_mw("single-pulse area, no drug", nodrug(W, "LID off"), nodrug(W, "unlesioned"), "off-state", "unlesioned")
for label, kw in [("whole-frame brightness", dict(roi="whole")), ("Fmin from tetrodotoxin only", dict(fmin_rule="ttx")), ("Fmin from the quinpirole baseline only", dict(fmin_rule="quin")), ("first two trials of each condition", dict(max_trials=2))]:
    V = grab_table(**kw); show_mw("variant, " + label, nodrug(V, "LID off"), nodrug(V, "unlesioned"), "off-state", "unlesioned")
print(W.groupby(["group", "date"])["No drug"].agg(["size", "median"]).round(3).to_string())

**Result: reproduced.** Off-state median 0.408 (11 regions) against unlesioned 0.111 (13 regions), P = 5.0e-05; every variant agrees.

### fig5f-ach-off-like-6ohda

> Evoked ACh release in the off-state is similar to that after a 6-OHDA lesion.

A claim of absence: two-sided Mann-Whitney, 6-OHDA against off-state; reproduced when P >= 0.05.

In [ ]:
show_mw("single-pulse area, no drug", nodrug(W, "6-OHDA"), nodrug(W, "LID off"), "6-OHDA", "off-state")
for label, kw in [("whole-frame brightness", dict(roi="whole")), ("Fmin from tetrodotoxin only", dict(fmin_rule="ttx")), ("Fmin from the quinpirole baseline only", dict(fmin_rule="quin")), ("first two trials of each condition", dict(max_trials=2))]:
    V = grab_table(**kw); show_mw("variant, " + label, nodrug(V, "6-OHDA"), nodrug(V, "LID off"), "6-OHDA", "off-state")

**Result: reproduced.** 6-OHDA median 0.315 (n = 10) against off-state 0.408 (n = 11), P = 0.245; the variants give P = 0.22 to 0.79.

### fig5f-ach-6ohda-elevated

> Evoked ACh release is markedly elevated in 6-OHDA-lesioned striatum compared with unlesioned striatum.

No criterion in the claims file: the paper's test, two-sided Mann-Whitney, P < 0.05 with 6-OHDA higher.

In [ ]:
show_mw("single-pulse area, no drug", nodrug(W, "6-OHDA"), nodrug(W, "unlesioned"), "6-OHDA", "unlesioned")
for label, kw in [("whole-frame brightness", dict(roi="whole")), ("Fmin from tetrodotoxin only", dict(fmin_rule="ttx")), ("Fmin from the quinpirole baseline only", dict(fmin_rule="quin")), ("first two trials of each condition", dict(max_trials=2))]:
    V = grab_table(**kw); show_mw("variant, " + label, nodrug(V, "6-OHDA"), nodrug(V, "unlesioned"), "6-OHDA", "unlesioned")

**Result: reproduced.** 6-OHDA median 0.315 against unlesioned 0.111, P = 0.00022.

### fig5f-ach-d2-control

> In unlesioned striatum sulpiride increases and quinpirole suppresses evoked ACh release.

No criterion in the claims file: the paper's paired test, two-sided Wilcoxon on the regions' single-pulse areas, P < 0.05 in the stated direction for both drugs.

In [ ]:
def paired(W, g, drug):
    d = W[W.group == g][["No drug", drug]].dropna()
    w = stats.wilcoxon(d[drug], d["No drug"])
    print(f"{g}, {drug} against no drug: n={len(d)} regions; median no drug {d['No drug'].median():.3f}, {drug} {d[drug].median():.3f}; "
          f"{drug} lower in {(d[drug] < d['No drug']).sum()}; Wilcoxon W={w.statistic:.1f}, P={w.pvalue:.4g}")
    return w.pvalue

paired(W, "unlesioned", "sulpiride"); paired(W, "unlesioned", "quinpirole")
for label, kw in [("whole-frame brightness", dict(roi="whole")), ("Fmin from tetrodotoxin only", dict(fmin_rule="ttx")), ("first two trials of each condition", dict(max_trials=2))]:
    V = grab_table(**kw); print("variant,", label); paired(V, "unlesioned", "sulpiride"); paired(V, "unlesioned", "quinpirole")

**Result: reproduced.** In the 13 unlesioned regions sulpiride raises the area in every region (median 0.111 to 0.243) and quinpirole lowers it in every region (to 0.036); Wilcoxon P = 0.00024 for both.

### fig5f-ach-da-suppresses

> Dopamine (50 nM) strongly suppresses evoked ACh release in 6-OHDA-lesioned and off-state striatum, and sulpiride has no effect after the lesion.

No criterion in the claims file: the paper's paired test, two-sided Wilcoxon; reproduced when dopamine lowers the area with P < 0.05 in both groups and sulpiride after the lesion has P >= 0.05.

In [ ]:
paired(W, "6-OHDA", "DA"); paired(W, "LID off", "DA"); paired(W, "6-OHDA", "sulpiride")
for label, kw in [("whole-frame brightness", dict(roi="whole")), ("Fmin from tetrodotoxin only", dict(fmin_rule="ttx")), ("first two trials of each condition", dict(max_trials=2))]:
    V = grab_table(**kw); print("variant,", label); paired(V, "6-OHDA", "DA"); paired(V, "LID off", "DA"); paired(V, "6-OHDA", "sulpiride")

**Result: reproduced.** Dopamine lowers the area in all 10 6-OHDA regions (0.315 to 0.103, P = 0.0020) and all 11 off-state regions (0.408 to 0.083, P = 0.00098); sulpiride after the lesion gives P = 0.92.

## AIMs

### link-aim-mice

> The dandiset's AIM score tables are mice of the paper's Figures 8 and 10, with genotypes and
> scores that agree with the paper's tables.

Mice are matched by their number. A session's total is the sum over its time points of the axial,
limb and orolingual scores.

In [ ]:
sc = aim[aim.behavior != "total"]
tot = sc.groupby(["figure", "group_label", "genotype", "mouse", "session"]).score.sum().rename("total").reset_index()
parts = sc.pivot_table(index=["figure", "group_label", "genotype", "mouse", "session"], columns="behavior", values="score", aggfunc="sum").reset_index()
labtot = aim[aim.behavior == "total"].groupby(["mouse", "session"]).score.sum().rename("lab_total").reset_index()
tot = tot.merge(labtot, on=["mouse", "session"]).merge(parts, on=["figure", "group_label", "genotype", "mouse", "session"])
print("time points per session:", aim.groupby(["mouse", "session"]).timestamp.nunique().value_counts().sort_index().to_dict())
print("sessions where the lab's total equals the sum of the three scores:", int((tot.total == tot.lab_total).sum()), "of", len(tot))

def aim_link(fig, sheet, gmap):
    P = T[sheet].set_axis(["mouse_p", "group", "session", "total_p"], axis=1)
    P["mouse"] = P.mouse_p.astype(str).str.extract(r"(\d+)")[0].astype(int)
    X = tot[tot.figure == fig].merge(P, on=["mouse", "session"], how="outer", indicator=True)
    X["label_agrees"] = X.group_label.map(gmap) == X.group
    return X

GMAP8 = {"control": "WT", "CDGI knockout": "KO"}
GMAP10 = {"control": "control", "M1R CRISPR": '"-M1R"'}
for fig, sheet, gm in [("Figure 8", "Fig 8B_total AIMs", GMAP8), ("Figure 10", "Fig 10B_total AIMs", GMAP10)]:
    X = aim_link(fig, sheet, gm)
    b = X[X._merge == "both"]
    print(f"\n{fig}: dandiset mice {X[X._merge != 'right_only'].mouse.nunique()}, paper mice {X[X._merge != 'left_only'].mouse.nunique()}, "
          f"matched {b.mouse.nunique()}; matched sessions {len(b)}, totals equal {(b.total == b.total_p).sum()}; "
          f"group labels agree for {b.drop_duplicates('mouse').label_agrees.sum()} of {b.mouse.nunique()} mice")
    print("dandiset mice not in the paper:", sorted(X[X._merge == "left_only"].mouse.unique().astype(int).tolist()),
          "; paper mice not in the dandiset:", sorted(X[X._merge == "right_only"].mouse.unique().astype(int).tolist()))
    print(f"median absolute difference over matched sessions {np.median(np.abs(b.total - b.total_p)):.2f}; largest {np.max(np.abs(b.total - b.total_p)):.1f}")
    print(b[b.total != b.total_p][["mouse", "group_label", "session", "total", "total_p"]].head(12).to_string())

**Result: not reproduced.** Figure 10 links: all 21 mice match by number, labels agree, and 103 of 105 session totals equal the paper's. Figure 8 does not: 18 of the 21 dandiset mice are in the paper (mice 4123, 4124 and 5871 are not; 2346, 2347, 7741, 7742 and 7744 of the paper are missing), labels agree for all 18, but only 12 of 90 matched session totals are equal (median absolute difference 2.5, largest 19.5).

### fig8bc-cdgi-aims

> Axial, limb, orolingual and total AIMs are less severe in CDGI KO mice than in wild type.

The claims file gives the paper's test, a repeated-measures two-way ANOVA (group by session); the
group effect is the between-mouse F of the mixed design, computed as a one-way ANOVA on each
mouse's mean over sessions (the same F with complete data). Reproduced when the group effect has
P < 0.05 with the KO lower, for the four scores. The scores are the dandiset's; the link above
says how they relate to the paper's.

In [ ]:
def aim_tests(fig, gmap, part_names=("total", "axial", "limb", "orolingual")):
    d = tot[tot.figure == fig]
    for part in part_names:
        r = zhai.mixed_anova_group(d, "mouse", "group_label", "session", part)
        means = d.groupby("group_label")[part].mean()
        print(f"{part}: group F({r['df1']},{r['df2']}) = {r['F']:.2f}, P = {r['p']:.3g}; mice {r['n']} with all {r['n_sessions']} sessions; "
              f"mean per session " + ", ".join(f"{k} {v:.1f}" for k, v in means.items()))

aim_tests("Figure 8", GMAP8)
pm8 = set(T["Fig 8B_total AIMs"].iloc[:, 0].astype(str).str.extract(r"(\d+)")[0].astype(int))
d8 = tot[(tot.figure == "Figure 8") & tot.mouse.isin(pm8)]
print(f"variant, only the {d8.mouse.nunique()} dandiset mice that are in the paper's table:")
for part in ["total", "axial", "limb", "orolingual"]:
    r = zhai.mixed_anova_group(d8, "mouse", "group_label", "session", part)
    means = d8.groupby("group_label")[part].mean()
    print(f"  {part}: F({r['df1']},{r['df2']}) = {r['F']:.2f}, P = {r['p']:.3g}; mean per session " + ", ".join(f"{k} {v:.1f}" for k, v in means.items()))
print("paper's table, same test:")
P = pd.concat([T[s].set_axis(["mouse", "group", "session", k], axis=1).set_index(["mouse", "group", "session"])
               for s, k in [("Fig 8B_total AIMs", "total"), ("Fig 8C_axial AIMs", "axial"), ("Fig 8C_limb AIMs", "limb"), ("Fig 8C_orolingual AIMs", "orolingual")]], axis=1).reset_index()
for part in ["total", "axial", "limb", "orolingual"]:
    r = zhai.mixed_anova_group(P, "mouse", "group", "session", part)
    print(f"  {part}: F({r['df1']},{r['df2']}) = {r['F']:.2f}, P = {r['p']:.3g}")

**Result: reproduced.** On the dandiset's scores (10 control and 11 CDGI KO mice, all with 5 sessions) the group effect has P = 2.2e-05 (total, F(1,19) = 31.2), 0.017 (axial), 6.4e-05 (limb) and 9.5e-06 (orolingual), with the KO lower in each. On the 18 mice that are also in the paper's table, P = 8.3e-06, 0.0077, 7.5e-05 and 9.9e-06. These are the lab's scoring sheets, which differ from the published Figure 8 (link-aim-mice); axial is weaker than in the paper (F = 6.87 against 19.91 on the paper's table).

### fig10b-m1r-aims

> AIMs are attenuated by deletion of M1Rs in iSPNs.

The same test as for Figure 8.

In [ ]:
def aim_tests(fig, gmap, part_names=("total", "axial", "limb", "orolingual")):
    d = tot[tot.figure == fig]
    for part in part_names:
        r = zhai.mixed_anova_group(d, "mouse", "group_label", "session", part)
        means = d.groupby("group_label")[part].mean()
        print(f"{part}: group F({r['df1']},{r['df2']}) = {r['F']:.2f}, P = {r['p']:.3g}; mice {r['n']} with all {r['n_sessions']} sessions; "
              f"mean per session " + ", ".join(f"{k} {v:.1f}" for k, v in means.items()))

aim_tests("Figure 10", GMAP10)

**Result: reproduced.** 11 control and 10 M1R CRISPR mice: group F(1,19) = 28.70 (total), 13.80 (axial), 41.43 (limb) and 44.58 (orolingual), all P < 0.0015 with the deletion lower; the paper reports 29.30, 13.80, 41.43 and 44.58.

## Claims on the paper's tables (data elsewhere)

These claims rest on groups the dandiset does not hold (control and 6-OHDA recordings,
rotations, open field, oxo-M line scans counted per dendrite). They are tested on the paper's own
per-cell and per-mouse tables (Zenodo 17307059), as the claims file says.

In [ ]:
def tcol(sheet, group, gcol=1):
    P = T[sheet]
    return P[P.iloc[:, gcol] == group].iloc[:, -1].to_numpy(float)
# The group labels of each sheet used below, with their counts
for s in ["Fig 1F_dSPN rheobase", "Fig 1I_dSPN dendritic excitabil", "Fig 3D_iSPN rheobase", "Fig 3F_iSPN dendritic excitabil",
          "Fig 7E_oxoM effect on DE", "Fig 8A_rotations 5th", "Fig 10A_rotations in M1R CRISPR", "Fig 10E_distance traveled", "Fig 10F_time mobile"]:
    print(f"{s}: {T[s].iloc[:, 1].value_counts(sort=False).to_dict()}")

### fig1-dspn-lesion-and-off-somatic

> dSPN somatic excitability (rheobase) rises after the 6-OHDA lesion and falls in the LID off-state.

Two-sided Mann-Whitney on the paper's Figure 1F table: lower rheobase after the lesion, higher in the off-state than after the lesion.

In [ ]:
s = "Fig 1F_dSPN rheobase"
show_mw("rheobase", tcol(s, "6-OHDA"), tcol(s, "control"), "6-OHDA", "control")
show_mw("rheobase", tcol(s, "LID off"), tcol(s, "6-OHDA"), "off-state", "6-OHDA");

**Result: reproduced.** On the paper's table, 6-OHDA rheobase median 120 pA against control 220 pA (P = 0.00030) and off-state 200 pA against 6-OHDA (P = 9.5e-05). The paper states P < 0.0001 for both; the first is 0.0003 here.

### fig1-dspn-lesion-dendritic-unchanged

> The dSPN dendritic excitability index is not changed by the 6-OHDA lesion or in the off-state.

A claim of absence on the paper's Figure 1I table; reproduced when both tests have P >= 0.05.

In [ ]:
s = "Fig 1I_dSPN dendritic excitabil"
show_mw("index", tcol(s, "control"), tcol(s, "6-OHDA"), "control", "6-OHDA")
show_mw("index", tcol(s, "6-OHDA"), tcol(s, "LID off"), "6-OHDA", "off-state");

**Result: reproduced.** P = 0.82 (control against 6-OHDA) and 0.90 (6-OHDA against off-state), as the paper reports (0.824 and 0.905).

### fig3-ispn-lesion-and-off-somatic

> iSPN somatic excitability is lowered by the 6-OHDA lesion and restored in the LID off-state.

Two-sided Mann-Whitney on the paper's Figure 3D table: higher rheobase after the lesion, lower in the off-state.

In [ ]:
s = "Fig 3D_iSPN rheobase"
show_mw("rheobase", tcol(s, "6-OHDA"), tcol(s, "Control"), "6-OHDA", "control")
show_mw("rheobase", tcol(s, "LID off"), tcol(s, "6-OHDA"), "off-state", "6-OHDA");

**Result: reproduced.** 6-OHDA median 200 pA against control 120 pA (P = 0.00040), off-state 130 pA against 6-OHDA (P = 0.00061).

### fig3-ispn-lesion-dendritic-up

> The iSPN dendritic excitability index rises after the lesion and stays raised in the off-state.

Two-sided Mann-Whitney on the paper's Figure 3F table, control against 6-OHDA and against off-state, the later group higher.

In [ ]:
s = "Fig 3F_iSPN dendritic excitabil"
show_mw("index", tcol(s, "6-OHDA"), tcol(s, "Control"), "6-OHDA", "control")
show_mw("index", tcol(s, "LID off"), tcol(s, "Control"), "off-state", "control");

**Result: reproduced.** 6-OHDA median 0.516 against control 0.405 (P = 0.035), off-state 0.519 against control (P = 0.024).

### fig7e-oxom-cdgi

> Oxotremorine-M increases bAP invasion of iSPN dendrites in wild-type but not in CDGI KO mice.

Two-sided Wilcoxon tests before against after oxo-M on the paper's Figure 7E table, rows paired by experiment number. Reproduced when wild type has P < 0.05 with oxo-M higher and KO has P >= 0.05. The dandiset's naive recordings are shown as a variant, one value per proximal/distal pair recorded in both conditions.

In [ ]:
P = T["Fig 7E_oxoM effect on DE"].set_axis(["expt", "genotype", "treatment", "index"], axis=1)
for g in ["WT", "KO"]:
    d = P[P.genotype == g].pivot_table(index="expt", columns="treatment", values="index")
    w = stats.wilcoxon(d["oxoM"], d["before (control)"])
    print(f"{g}: n={len(d)} dendrites; median before {d['before (control)'].median():.3f}, oxo-M {d['oxoM'].median():.3f}; "
          f"oxo-M higher in {(d['oxoM'] > d['before (control)']).sum()}; Wilcoxon W={w.statistic:.1f}, P={w.pvalue:.4g}")
for g in ["WT", "KO"]:
    d = P[P.genotype == g].pivot_table(index="expt", columns="treatment", values="index")
    print(f"variant, {g} unpaired:", end=" "); show_mw("Mann-Whitney", d["oxoM"], d["before (control)"], "oxo-M", "before")
# variant on the dandiset's own naive line scans, per proximal/distal pair
dp = den_pairs[den_pairs.condition.str.contains("naive")].copy()
dp["oxo"] = dp.condition.str.contains("oxo")
dp["genotype"] = np.where(dp.condition.str.contains("CDGI"), "KO", "WT")
dp["pairnum"] = dp.proximal.str.extract(r"(\d+)")[0]
v = dp.pivot_table(index=["genotype", "path", "cell", "pairnum"], columns="oxo", values="index").dropna()
for g in ["WT", "KO"]:
    d = v.loc[g]
    w = stats.wilcoxon(d[True], d[False])
    print(f"variant, dandiset {g} pairs recorded before and with oxo-M: n={len(d)}; median {d[False].median():.3f} before, {d[True].median():.3f} with; Wilcoxon P={w.pvalue:.4g}")

**Result: reproduced.** Wild type: oxo-M raises the index in 8 of 10 dendrites, Wilcoxon P = 0.0098; KO: P = 0.47 (the paper reports 0.910). The table pairs rows by experiment number (WT_1 to WT_10, KO_1 to KO_12), so the pairing is not open; an unpaired Mann-Whitney test gives KO P = 0.89 and wild type P = 0.054. The dandiset's own naive pairs agree with the paired result (WT P = 0.0039, KO P = 0.38).

### fig8a-cdgi-rotations

> Levodopa induces more contralateral rotations in CDGI KO mice than in wild type at the fifth dose.

Two-sided Mann-Whitney on the paper's per-mouse table, KO higher.

In [ ]:
s = "Fig 8A_rotations 5th"
show_mw("rotations in 30 s", tcol(s, "KO"), tcol(s, "WT"), "KO", "WT");

**Result: reproduced.** KO median 8.5 rotations (n = 10) against wild type 4.25 (n = 12), P = 0.012 (the paper's text gives 0.0092).

### fig10a-m1r-rotations

> Levodopa induces more contralateral rotations in mice lacking M1Rs in iSPNs.

Two-sided Mann-Whitney on the paper's per-mouse table, deletion higher.

In [ ]:
s = "Fig 10A_rotations in M1R CRISPR"
show_mw("rotations in 30 s", tcol(s, '"-M1R"'), tcol(s, "control"), "M1R deletion", "control");

**Result: reproduced.** M1R deletion median 8.5 (n = 10) against control 6.0 (n = 11), P = 0.0078 (paper 0.0056).

### fig10ef-open-field

> In MCI-Park mice, a low dose of levodopa increases locomotion profoundly only when M1Rs are deleted from the striatum; M1R deletion alone does not.

Two-sided Mann-Whitney tests on the paper's per-mouse tables, for distance and time mobile.

In [ ]:
for s in ["Fig 10E_distance traveled", "Fig 10F_time mobile"]:
    print(s)
    show_mw("  control against MCI-Park", tcol(s, "control"), tcol(s, "MCI-Park"), "control", "MCI-Park")
    show_mw("  levodopa with against without M1R deletion", tcol(s, '"+L-dopa, -M1R"'), tcol(s, '"+L-dopa"'), "+L-dopa -M1R", "+L-dopa")
    show_mw("  MCI-Park against M1R deletion alone", tcol(s, "MCI-Park"), tcol(s, '"-M1R"'), "MCI-Park", "-M1R")

**Result: reproduced.** Control against MCI-Park P = 1.1e-05 (distance) and 7.0e-06 (time mobile); levodopa with against without M1R deletion P = 0.00033 and 0.00057, the deletion higher; MCI-Park against deletion alone P = 0.63 and 0.47. The sheet's "-M1R" group (9 mice) is MCI-Park with M1R deletion, as the Figure 10 legend names it (n = 9), and these two P values equal the paper's 0.6340 and 0.4657.

## Blocked claims

- fig2bc-dspn-spines, fig4bc-ispn-spines, fig6f-m1r-antag-spines, fig7h-cdgi-ko-spines,
  fig9f-m1r-crispr-spines: the dandiset holds the raw 2PLSM z-stacks (cell `finding-missing-groups`
  counts the files by cell type and group, and lists what one file of each holds: image stacks
  under `acquisition`, and nothing under `processing` or `analysis`), so no spine counts,
  annotations or segmentations. The paper deconvolved the stacks in AutoQuant X3 and counted
  spines semi-automatically in NeuronStudio, with mushroom spines classified there; neither can
  be run here. Not runnable. The paper's own spine tables are in its Zenodo deposit, but re-running
  its statistics on them would not test the dandiset.
- fig4k-confocal-spines: the confocal images of sparsely labelled iSPNs are not in the dandiset.
  They are in the paper's raw-data deposit on Zenodo (record 17260198, one 13.4 GB zip), and the
  paper counted spines in Imaris 10 with supervised learning and manual correction, which cannot
  be run here. Not runnable.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| link-somatic-cells | partially reproduced (90 of 123 cells exact) |
| link-dendritic-cells | reproduced |
| link-sr-cells | partially reproduced (21 of 33 within 25%) |
| link-grab-rois | not reproduced (16 of 34 regions within 0.05; 1 of 11 mice) |
| link-aim-mice | not reproduced (Figure 8 scores differ) |
| fig1f-dspn-rheobase-on-lower, fig1e-dspn-fi-on-higher, fig1f-dspn-sch-no-change | reproduced |
| fig1i-dspn-dendritic-on-higher | reproduced |
| fig1i-dspn-dendritic-sch-lower | not reproduced (P = 0.066) |
| fig1-dspn-lesion-and-off-somatic, fig1-dspn-lesion-dendritic-unchanged | reproduced (paper's tables) |
| fig2h, fig2i, fig4g, fig4h (Sr2+-oEPSC) | reproduced, resting on the stand-in detector (fig4h also on which 10 sweeps of two cells are used) |
| fig3d-ispn-rheobase-on-higher, fig3c-ispn-fi-on-lower, fig3d-ispn-sul-no-change | reproduced |
| fig3f-ispn-dendritic-on-lower, fig3f-ispn-dendritic-sul-no-change | reproduced |
| fig3-ispn-lesion-and-off-somatic, fig3-ispn-lesion-dendritic-up | reproduced (paper's tables) |
| fig5f (five GRAB-ACh claims) | reproduced |
| fig6c, fig6d, fig7c, fig7f, fig9d | reproduced |
| fig7e-oxom-cdgi, fig8a, fig10a, fig10ef | reproduced (paper's tables) |
| fig8bc-cdgi-aims, fig10b-m1r-aims | reproduced |
| fig2bc, fig4bc, fig6f, fig7h, fig9f (2PLSM spines), fig4k (confocal) | blocked, not runnable |

How the analysis differs from the paper's: spikes, rheobase, the dendritic index and the GRAB
areas follow the paper's scripts, but from the files' data (the GRAB ROI is the 50 um circle
computed from the movies; background images are not shared, which cancels in the normalised
area). Sr2+-oEPSC events come from a stand-in detector, not TaroTools with visual checking. The
AIM group effect is the between-mouse F of the mixed design. Eight claims rest on the paper's
own tables, as the claims file says. After the first verification round the red-channel background
of the dendritic index was corrected (55, where 161 had been subtracted from both channels), the
last-sweep variant of the somatic measures was made to take the last sweep, and the Sr2+ claims
gained the last-10 and nearest-run sweep variants; no outcome changed. After the second round every
somatic test was also run under the last sweep of a repeated step (cell `somatic-last-sweep`).

What was found about the dataset: it holds every cell of the paper's off-state, on-state and drug
groups for somatic, dendritic and Sr2+ recordings, and every GRAB region; the control and
6-OHDA patch-clamp recordings, the open-field data and the confocal images are not in it.
33 of 123 somatic cells do not match the paper's spike counts exactly. One dendritic cell lacks
the pair of sites the paper used, which changes the outcome of fig1i-dspn-dendritic-sch-lower
(P = 0.066 with it, 0.027 without it).
The Figure 8 AIM scores are the lab's sheets and differ from the published figure. Two Sr2+
cells hold more sweeps than the paper used, and several GRAB regions hold more trials than the
two the Methods give and lack a full calibration trial.